In [1]:
import os
import cv2
import numpy as np
import os
import time

In [2]:
# ─── Rutas ────────────────────────────────────────────────────────────────────
SCRIPT_DIR = os.getcwd()

RESULTS_DIR = os.path.join(SCRIPT_DIR, "..", "resultados")
os.makedirs(RESULTS_DIR, exist_ok=True)

def guardar(nombre: str, imagen: np.ndarray) -> None:
    """Guarda una imagen en la carpeta de resultados."""
    ruta = os.path.join(RESULTS_DIR, nombre)
    cv2.imwrite(ruta, imagen)
    print(f"  [✓] Guardado: {nombre}")

In [3]:
# ─── PASO 1: Cargar entrada visual ────────────────────────────────────────────
print("\n[1] Cargando imagen de entrada...")

# Se genera una imagen sintética representativa en caso de no haber una imagen externa.
# Parámetro: 480×640 px, 3 canales BGR.
# Justificación: garantiza reproducibilidad sin dependencias externas.
altura, ancho = 480, 640
imagen_original = np.zeros((altura, ancho, 3), dtype=np.uint8)

# Fondo degradado azul-verde
for y in range(altura):
    for x in range(ancho):
        imagen_original[y, x] = [
            int(80 + 120 * x / ancho),     # B
            int(50 + 100 * y / altura),     # G
            int(30 + 60 * x / ancho),       # R
        ]


[1] Cargando imagen de entrada...


In [4]:
# Círculos de colores (objetos detectables)
cv2.circle(imagen_original, (160, 160), 90, (220, 60,  40),  -1)
cv2.circle(imagen_original, (480, 160), 70, (40,  200, 60),  -1)
cv2.circle(imagen_original, (160, 340), 60, (60,  60,  220), -1)
cv2.circle(imagen_original, (480, 320), 80, (200, 180, 40),  -1)

# Rectángulos (objetos adicionales)
cv2.rectangle(imagen_original, (260, 200), (380, 300), (220, 100, 200), -1)
cv2.rectangle(imagen_original, (270, 380), (370, 440), (40,  200, 200), -1)

# Texto sobre la imagen
cv2.putText(imagen_original, "Procesamiento Visual", (90, 460),
            cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2)

guardar("original.png", imagen_original)
print(f"     Dimensiones: {imagen_original.shape[1]}×{imagen_original.shape[0]} px")

  [✓] Guardado: original.png
     Dimensiones: 640×480 px


In [5]:
# ─── PASO 2: Escala de grises ─────────────────────────────────────────────────
print("\n[2] Convirtiendo a escala de grises...")
# Parámetro: conversión estándar BGR→GRAY usando pesos perceptuales (ITU-R 601).
# Justificación: reduce dimensionalidad manteniendo información de luminancia.
grises = cv2.cvtColor(imagen_original, cv2.COLOR_BGR2GRAY)
guardar("grises.png", grises)


[2] Convirtiendo a escala de grises...
  [✓] Guardado: grises.png


In [6]:
# ─── PASO 3: Espacio de color HSV ────────────────────────────────────────────
print("\n[3] Convirtiendo a espacio HSV...")
# Parámetro: espacio HSV (Hue 0–179, Saturation 0–255, Value 0–255).
# Justificación: HSV desacopla la información de color (H) de la luminosidad (V),
# lo que facilita la segmentación posterior por color independientemente de la
# iluminación.
hsv = cv2.cvtColor(imagen_original, cv2.COLOR_BGR2HSV)
guardar("hsv.png", hsv)


[3] Convirtiendo a espacio HSV...
  [✓] Guardado: hsv.png


In [7]:
# ─── PASO 4: Suavizado Gaussiano ──────────────────────────────────────────────
print("\n[4] Aplicando suavizado Gaussiano...")
# Parámetro: kernel 15×15, σ=0 (calculado automáticamente ~4.0).
# Justificación: un kernel grande elimina ruido de alta frecuencia sin perder
# bordes gruesos; σ automático es suficiente para esta resolución.
suavizado = cv2.GaussianBlur(imagen_original, (15, 15), 0)
guardar("suavizado.png", suavizado)


[4] Aplicando suavizado Gaussiano...
  [✓] Guardado: suavizado.png


In [8]:
# ─── PASO 5: Detección de bordes Canny ───────────────────────────────────────
print("\n[5] Detectando bordes con Canny...")
# Parámetros: threshold1=50, threshold2=150 (ratio 1:3 recomendado por Canny).
# Justificación: el umbral bajo captura bordes débiles conectados a bordes fuertes;
# el ratio 1:3 equilibra sensibilidad y eliminación de ruido.
gris_suavizado = cv2.cvtColor(suavizado, cv2.COLOR_BGR2GRAY)
bordes = cv2.Canny(gris_suavizado, threshold1=50, threshold2=150)
guardar("bordes.png", bordes)


[5] Detectando bordes con Canny...
  [✓] Guardado: bordes.png


In [9]:
# ─── PASO 6: Segmentación / Detección de objetos ─────────────────────────────
print("\n[6] Segmentando y detectando objetos (técnica clásica)...")

# Técnica: Umbralización en HSV + búsqueda de contornos.
# Justificación: la segmentación por color en HSV es robusta ante variaciones
# de iluminación y adecuada para objetos de colores definidos. Los contornos
# permiten obtener bounding boxes sin modelos preentrenados.

resultado_deteccion = imagen_original.copy()
objetos_encontrados = 0

# Rangos de colores a detectar (H, S, V mínimo y máximo)
rangos_colores = {
    "Rojo":     ([0,   100, 80],  [10,  255, 255]),
    "Verde":    ([45,  80,  80],  [85,  255, 255]),
    "Azul":     ([100, 80,  80],  [130, 255, 255]),
    "Amarillo": ([20,  80,  80],  [40,  255, 255]),
    "Magenta":  ([140, 60,  80],  [170, 255, 255]),
    "Cyan":     ([85,  80,  80],  [100, 255, 255]),
}

colores_bbox = {
    "Rojo":     (0,   0,   255),
    "Verde":    (0,   255, 0  ),
    "Azul":     (255, 0,   0  ),
    "Amarillo": (0,   255, 255),
    "Magenta":  (255, 0,   255),
    "Cyan":     (255, 255, 0  ),
}

for nombre_color, (bajo, alto) in rangos_colores.items():
    bajo_np = np.array(bajo,  dtype=np.uint8)
    alto_np = np.array(alto,  dtype=np.uint8)

    mascara  = cv2.inRange(hsv, bajo_np, alto_np)
    contornos, _ = cv2.findContours(mascara, cv2.RETR_EXTERNAL,
                                    cv2.CHAIN_APPROX_SIMPLE)

    for contorno in contornos:
        area = cv2.contourArea(contorno)
        if area < 1500:          # Parámetro: área mínima 1500 px²
            continue             # Descarta ruido pequeño
        objetos_encontrados += 1
        x, y, w, h = cv2.boundingRect(contorno)
        color_bbox  = colores_bbox[nombre_color]
        cv2.rectangle(resultado_deteccion, (x, y), (x+w, y+h), color_bbox, 2)
        cv2.putText(resultado_deteccion, nombre_color,
                    (x, max(y - 8, 16)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, color_bbox, 2)

cv2.putText(resultado_deteccion,
            f"Objetos detectados: {objetos_encontrados}",
            (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

guardar("deteccion_segmentacion.png", resultado_deteccion)
print(f"     Objetos detectados: {objetos_encontrados}")


[6] Segmentando y detectando objetos (técnica clásica)...
  [✓] Guardado: deteccion_segmentacion.png
     Objetos detectados: 7


In [10]:
# ─── PASO 7: Panel comparativo ───────────────────────────────────────────────
print("\n[7] Generando panel comparativo...")

def a_bgr_3ch(img: np.ndarray) -> np.ndarray:
    """Convierte cualquier imagen a BGR 3 canales."""
    if img.ndim == 2:
        return cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    return img.copy()

ANCHO_PANEL = 320
ALTO_PANEL  = 240

imagenes_panel = [
    ("Original",     imagen_original),
    ("Grises",       grises),
    ("HSV",          hsv),
    ("Suavizado",    suavizado),
    ("Bordes",       bordes),
    ("Deteccion",    resultado_deteccion),
]

celdas = []
for titulo, img in imagenes_panel:
    celda = cv2.resize(a_bgr_3ch(img), (ANCHO_PANEL, ALTO_PANEL))
    cv2.rectangle(celda, (0, 0), (ANCHO_PANEL - 1, ALTO_PANEL - 1), (200, 200, 200), 1)
    cv2.putText(celda, titulo, (8, 22),
                cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 255), 2)
    cv2.putText(celda, titulo, (8, 22),
                cv2.FONT_HERSHEY_SIMPLEX, 0.65, (30,  30,  30),  1)
    celdas.append(celda)

fila1 = np.hstack(celdas[:3])
fila2 = np.hstack(celdas[3:])
panel = np.vstack([fila1, fila2])
guardar("panel_comparativo.png", panel)


[7] Generando panel comparativo...
  [✓] Guardado: panel_comparativo.png


In [11]:
# ─── Resumen ──────────────────────────────────────────────────────────────────
print("\n" + "─" * 55)
print("  PROCESAMIENTO COMPLETADO")
print("─" * 55)
archivos = sorted(os.listdir(RESULTS_DIR))
for f in archivos:
    ruta_f = os.path.join(RESULTS_DIR, f)
    kb = os.path.getsize(ruta_f) / 1024
    print(f"  {f:<38} {kb:6.1f} KB")
print("─" * 55)
print(f"  Total archivos generados: {len(archivos)}")
print(f"  Carpeta: {os.path.abspath(RESULTS_DIR)}\n")


───────────────────────────────────────────────────────
  PROCESAMIENTO COMPLETADO
───────────────────────────────────────────────────────
  bordes.png                                2.0 KB
  deteccion_segmentacion.png               86.8 KB
  grises.png                               19.0 KB
  hsv.png                                  71.3 KB
  original.png                             66.2 KB
  panel_comparativo.png                   204.6 KB
  suavizado.png                           121.1 KB
───────────────────────────────────────────────────────
  Total archivos generados: 7
  Carpeta: E:\Users\famil\Desktop\RepoFinal\examen-final-computacion-visual-jose-herrera\ejercicio_1_procesamiento_visual\resultados

